# Session 2 - Importing Data with Python

Welcome! Before you can analyse data, you have to **get it into Python**. In this notebook we'll practise three common ways:

| Part | Where the data lives | Tools we use |
|---|---|---|
| 1 | A **local file** on your computer | `os` + `pandas` |
| 2 | An **API** on the internet | `requests` + `pandas` |
| 3 | **Google Drive** | `google.colab` (mount) or a share link + `pandas` |

> The script `importing_data.py` in this same folder does exactly the same thing, but runs everything at once. Compare them when you finish!

### How to use a notebook
- A notebook is made of **cells**. This one is a *text* cell (Markdown). Grey boxes with code are *code* cells.
- To run a code cell, click on it and press **Shift + Enter**. The result appears just below it.
- Run the cells **in order, from top to bottom**. Later cells use things created in earlier ones.
- If something gets weird, use **Restart** (top toolbar) and run everything again from the start.
- Lines that start with `#` are **comments**: notes for humans. Python ignores them.

## Step 0 - Install and import the libraries

A **library** is a toolbox of code that someone else already wrote. We need two that don't come with Python:
- **pandas** - for working with tables of data
- **requests** - for talking to websites and APIs

The cell below installs them. You only need to do this **once** per computer (Google Colab already has them, so you can skip it there).

In [1]:
# The "%" at the start is a notebook "magic" command: it runs pip
# (Python's package installer) for the Python used by this notebook.
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\leodo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# "import" loads a library so we can use it.
import os            # "operating system": works with folders and file paths
import pandas as pd  # tables of data. "pd" is a short nickname everybody uses
import requests      # sends requests to websites and APIs

# If this cell runs without a red error, everything is installed correctly!
print("pandas version:", pd.__version__)

pandas version: 2.3.3


---
## Part 1 - Loading a local file (`os` + `pandas`)

We'll load `data/students_sample.csv`, a small made-up dataset included in this folder.

A **CSV** (*Comma-Separated Values*) file is a table saved as plain text: each line is a row, and commas separate the columns. It looks like this inside:

```
student_id,name,group,hours_studied,exam_score,attended_sessions
1,Ana,Foundations,4.5,6.8,5
2,Bruno,Advanced,7.0,8.9,6
```

### 1.1 Where am I?
Every program runs *inside* a folder, called the **working directory**. When you give Python just a file name, it looks for it there.

In [3]:
# os.getcwd() = "get current working directory"
print(os.getcwd())

c:\Users\leodo\Desktop\Club\STUDENTS-repo\DOCUMENTATION\03-session material\session 2


In [4]:
# os.listdir() returns a LIST with the names of everything in a folder.
# "." means "the current folder".
print(os.listdir("."))

['data', 'importing_data.ipynb', 'importing_data.py']


### 1.2 Building a file path
A **path** is the address of a file. Windows separates folders with `\`, Mac and Linux use `/`.

To avoid problems, never type the slashes yourself: use **`os.path.join()`**, which picks the right one for your computer.

In [5]:
data_folder = os.path.join(".", "data")
csv_path = os.path.join(data_folder, "students_sample.csv")

print("Data folder:", data_folder)
print("File path:  ", csv_path)

Data folder: .\data
File path:   .\data\students_sample.csv


In [6]:
# os.path.exists() returns True or False.
# Checking first lets us print a friendly message instead of a scary error.
if os.path.exists(csv_path):
    print("The file exists - let's load it!")
else:
    print("File not found.")
    print("Make sure this notebook is in the same folder as the 'data' folder.")
    print("On Google Colab: click the folder icon on the left and upload the file.")

The file exists - let's load it!


### 1.3 Reading the CSV with pandas
`pd.read_csv()` reads the file and turns it into a **DataFrame**: pandas' name for a table with rows and columns, just like a sheet in Excel.

In a notebook, writing a variable's name on the **last line** of a cell displays it nicely - no `print()` needed.

In [7]:
students = pd.read_csv(csv_path)

students   # show the whole table

,student_id,name,group,hours_studied,exam_score,attended_sessions
0,1,Ana,Foundations,4.5,6.8,5
1,2,Bruno,Advanced,7.0,8.9,6
2,3,Carla,Foundations,2.0,5.1,3
3,4,Diego,Advanced,5.5,7.4,6
4,5,Elena,Foundations,6.0,8.2,6
5,6,Fernando,Advanced,3.0,5.9,4
6,7,Gabriela,Foundations,8.0,9.3,6
7,8,Hugo,Advanced,1.5,4.2,2
8,9,Irene,Foundations,5.0,7.0,5
9,10,Javier,Advanced,6.5,8.5,5


### 1.4 First look at the data
These commands are the first thing you should run on **any** new dataset:

| Command | What it tells you |
|---|---|
| `.head()` | the first 5 rows |
| `.shape` | (number of rows, number of columns) |
| `.columns` | the column names |
| `.info()` | the type of each column and whether values are missing |
| `.describe()` | quick statistics for the number columns |

In [ ]:
students.head()

In [ ]:
print("Shape (rows, columns):", students.shape)
print("Columns:", list(students.columns))

In [ ]:
# Data types: int64 = whole numbers, float64 = decimals, object = text.
# "non-null" counts the values that are NOT missing.
students.info()

In [ ]:
# count, mean (average), std (how spread out the values are),
# min, max and the quartiles (25%, 50% = median, 75%).
students.describe()

### 1.5 A first tiny analysis
`groupby("group")` splits the table by the value in the `group` column. Then `.mean()` calculates the average for each part.

In [ ]:
students.groupby("group")["exam_score"].mean()

> **Try it yourself:** change `"exam_score"` to `"hours_studied"` and run the cell again.

**Other file types** load almost the same way:
- `pd.read_excel("file.xlsx")` - Excel (needs `pip install openpyxl`)
- `pd.read_json("file.json")` - JSON
- `pd.read_csv("file.csv", sep=";")` - CSV files that use `;` instead of `,`

---
## Part 2 - Loading data from an API (`requests` + `pandas`)

An **API** (*Application Programming Interface*) is a way for programs to ask a website for data. Instead of a web page made for humans, the API answers with raw data - usually in a format called **JSON**.

Think of it like a restaurant: you (the program) give your order (the **request**) to the waiter (the API), and the kitchen (the server) sends back your food (the **response**).

We'll use [Open-Meteo](https://open-meteo.com), a free weather API that doesn't need an account or a password (an *API key*). We'll ask for the weather in Pamplona over the last 7 days.

### 2.1 Prepare the request

In [8]:
# WHERE we send the question
api_url = "https://api.open-meteo.com/v1/forecast"

# WHAT we ask for. This is a DICTIONARY: "name": value pairs inside { }.
params = {
    "latitude": 42.81,          # Pamplona's position on the map
    "longitude": -1.65,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Europe/Madrid",
    "past_days": 7,             # the last 7 days...
    "forecast_days": 1,         # ...plus today
}

### 2.2 Send it and check the answer
Every response comes with a **status code**:
- `200` - OK, everything worked
- `404` - not found (probably a typo in the URL)
- `500` - the server had a problem

In [9]:
# timeout=10 means "give up if there's no answer after 10 seconds"
response = requests.get(api_url, params=params, timeout=10)

print("Status code:", response.status_code)
print("Full URL we called:", response.url)

Status code: 200
Full URL we called: https://api.open-meteo.com/v1/forecast?latitude=42.81&longitude=-1.65&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Europe%2FMadrid&past_days=7&forecast_days=1


> Copy the **Full URL** printed above and paste it into your browser. You'll see exactly the same JSON data Python received!

In [10]:
# .json() converts the text answer into Python dictionaries and lists
data = response.json()

# Which "keys" (labels) does the answer have?
list(data.keys())

['latitude',
 'longitude',
 'generationtime_ms',
 'utc_offset_seconds',
 'timezone',
 'timezone_abbreviation',
 'elevation',
 'daily_units',
 'daily']

In [11]:
# The weather values live under the "daily" key.
# It's a dictionary of lists: {"time": [...], "temperature_2m_max": [...], ...}
data["daily"]

{'time': ['2026-10-29',
  '2026-10-30',
  '2026-10-31',
  '2026-11-01',
  '2026-11-02',
  '2026-11-03',
  '2026-11-04',
  '2026-11-05'],
 'temperature_2m_max': [20.0, 20.7, 19.1, 16.4, 15.5, 10.8, 13.1, 14.9],
 'temperature_2m_min': [7.4, 9.1, 6.8, 5.4, 8.0, 4.7, 4.0, 2.8],
 'precipitation_sum': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}

### 2.3 Turn the JSON into a DataFrame
A dictionary of lists like this one converts straight into a table: each key becomes a column.

In [ ]:
weather = pd.DataFrame(data["daily"])

# Give the columns friendlier names
weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_max": "max_temp_C",
    "temperature_2m_min": "min_temp_C",
    "precipitation_sum": "rain_mm",
})

# The dates arrive as text. pd.to_datetime() turns them into real dates,
# so pandas can sort, filter and plot them correctly.
weather["date"] = pd.to_datetime(weather["date"])

weather

In [ ]:
print("Average max temperature:", round(weather["max_temp_C"].mean(), 1), "C")
print("Coldest night:", weather["min_temp_C"].min(), "C")

### 2.4 Save a copy
Saving to a file means you don't need to call the API again next time. `index=False` stops pandas from writing the row numbers as an extra column.

In [ ]:
output_path = os.path.join(data_folder, "pamplona_weather.csv")
weather.to_csv(output_path, index=False)

print("Saved to:", output_path)

> **Try it yourself:** look up the latitude and longitude of your home town (search it on Google Maps), change them in step 2.1, and run Part 2 again.

---
## Part 3 - Loading data from Google Drive

There are two common ways to read a file stored in your Google Drive.

### Option A - Mount your Drive (only in Google Colab)
[Google Colab](https://colab.research.google.com) is a free website that runs notebooks online. In Colab you can **mount** your Google Drive: your whole Drive appears as a normal folder, and you read files with `pd.read_csv()` exactly like in Part 1.

**This only works inside Colab.** In VS Code or Jupyter on your own computer, the cell below just prints a message - use Option B instead.

When you run it in Colab, a window will ask you to sign in and give permission. That's normal.

In [12]:
try:
    # This library only exists inside Google Colab
    from google.colab import drive

    # "Mount" = connect your Drive at the folder /content/drive
    drive.mount("/content/drive")

    # Your files are now inside /content/drive/MyDrive
    drive_folder = "/content/drive/MyDrive"
    print("Drive mounted! A few of your files:")
    print(os.listdir(drive_folder)[:10])   # [:10] = only the first 10

except ImportError:
    # ImportError happens when Python can't find a library
    print("Not running in Google Colab - skipping the mount.")
    print("Use Option B below instead.")

Not running in Google Colab - skipping the mount.
Use Option B below instead.


Once mounted, change the path below to point to a CSV in your Drive and run it (in Colab only).

**Tip:** in Colab's left panel, open the folder icon, find your file, click the three dots and choose **Copy path**.

In [ ]:
# Change this to the path of your own file
my_drive_file = "/content/drive/MyDrive/students_sample.csv"

if os.path.exists(my_drive_file):
    drive_data = pd.read_csv(my_drive_file)
    display(drive_data.head())   # display() shows a table nicely, like the last line of a cell
else:
    print("File not found:", my_drive_file)
    print("(Normal if you're not in Colab or haven't changed the path.)")

### Option B - Use a share link (works everywhere)
`pd.read_csv()` can read straight from a web address, not only from files on your computer. So if a CSV is shared publicly, we can load it from anywhere.

**How to get the file ID:**
1. In Google Drive, right-click your CSV file -> **Share**.
2. Under *General access* choose **Anyone with the link**.
3. Click **Copy link**. It looks like:
   `https://drive.google.com/file/d/1AbCdEfGhIjKlMnOp/view?usp=sharing`
4. The **file ID** is the code between `/d/` and `/view`: `1AbCdEfGhIjKlMnOp`
5. Paste it in the cell below.

> **Warning:** "Anyone with the link" means *anyone*. Never share files with personal or private data this way.

In [ ]:
DRIVE_FILE_ID = ""   # <-- paste your file ID between the quotes

# An empty text "" counts as False, so this checks "did you paste an ID?"
if DRIVE_FILE_ID:
    # This special URL asks Drive for the raw file instead of the preview page.
    # The f"..." (an "f-string") puts the value of DRIVE_FILE_ID where the { } are.
    drive_url = f"https://drive.google.com/uc?export=download&id={DRIVE_FILE_ID}"

    try:
        shared_data = pd.read_csv(drive_url)
        display(shared_data.head())
        print("Shape (rows, columns):", shared_data.shape)
    except Exception as error:
        print("Could not read the file. Check that:")
        print("  - the ID is correct")
        print("  - the file is shared as 'Anyone with the link'")
        print("  - the file is a CSV (not a Google Sheet or Excel file)")
        print("Details:", error)
else:
    print("No file ID yet - paste one into DRIVE_FILE_ID and run this cell again.")

---
## Summary

| You want to... | Use |
|---|---|
| know which folder you're in | `os.getcwd()` |
| see the files in a folder | `os.listdir()` |
| build a path safely | `os.path.join()` |
| check a file exists | `os.path.exists()` |
| load a CSV (from a file *or* a link) | `pd.read_csv()` |
| get data from an API | `requests.get()` + `.json()` |
| turn JSON into a table | `pd.DataFrame()` |
| take a first look | `.head()`, `.shape`, `.info()`, `.describe()` |
| save a table | `.to_csv()` |

### Challenges
1. Load `data/pamplona_weather.csv` (the file you saved in Part 2) back in with `pd.read_csv()`.
2. In Part 1, find the student who studied the most hours. *Hint:* `students.sort_values("hours_studied")`
3. Upload `students_sample.csv` to your Google Drive and load it with Option A (in Colab) **and** Option B.
4. Open `importing_data.py` and run it from a terminal with `python importing_data.py`. What's the difference between running a script and running a notebook?